# Lab 5: Linear Algebra Essentials

> **Course:** Data Science Foundations
> **Lesson:** Day 5 Lab, companion to `5-linear-algebra-essentials.md`
> **Dataset:** `mpg` (seaborn), reused from Day 1 so the focus stays on the maths
> **Estimated time:** 2 hours

## Lab Objectives

- Do vector operations by hand, then verify them in NumPy, including angles and orthogonality
- Implement matrix multiplication with loops and confirm the dimension rule
- Check key matrix properties: non-commutativity, the transpose of a product, the identity, the **inverse**, and singular matrices
- Use **cosine similarity** on standardised data, and see why raw dot products mislead
- Build a covariance matrix with one matrix product
- Fit linear regression with the normal equations the right way: split before scaling, and `solve` instead of `inv`
- See how **collinear** features break $X^TX$

**How to work through this lab**

- Cells marked `# TODO` are yours to complete. Replace each `None` or comment with working code.
- Run every **Checkpoint** cell. If it prints `passed`, move on. If it raises an error, fix your code first.
- Each **Question** needs a short written answer in the "Your answer" cell below it.
- The dataset is different from the lesson on purpose. Adapt the lesson code; don't paste it.
- Worked solutions are in the `solutions/` folder. Attempt every part yourself before you open them: the struggle is where the learning happens.

## Part 0: Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)

cars = sns.load_dataset("mpg").dropna().reset_index(drop=True)
print(f"Shape: {cars.shape}")

## Part 1: Vectors by Hand

$$\vec{u} = \begin{bmatrix} 3 \\ -1 \\ 2 \end{bmatrix}, \qquad \vec{v} = \begin{bmatrix} 1 \\ 4 \\ 0 \end{bmatrix}, \qquad \vec{w} = \begin{bmatrix} 2 \\ 2 \\ -2 \end{bmatrix}$$

Work these out **on paper first**, then type your answers as plain numbers (no NumPy) in the cell below. The checkpoint compares them with NumPy.

In [ ]:
by_hand = {
    "u + v": None,              # a list of 3 numbers
    "2u - v": None,             # a list of 3 numbers
    "u . v": None,              # one number
    "||u||": None,              # one number (you may leave it as 14 ** 0.5)
    "u . w": None,              # one number
}

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
u, v, w = np.array([3, -1, 2]), np.array([1, 4, 0]), np.array([2, 2, -2])
assert np.allclose(by_hand["u + v"], u + v)
assert np.allclose(by_hand["2u - v"], 2 * u - v)
assert np.isclose(by_hand["u . v"], u @ v)
assert np.isclose(by_hand["||u||"], np.linalg.norm(u))
assert np.isclose(by_hand["u . w"], u @ w)
print("passed")

In [ ]:
# TODO 1.2: the angle between u and v in degrees, using  u.v = ||u|| ||v|| cos(theta)
angle_uv = None
print(angle_uv)

**Question 1.** What does $\vec{u} \cdot \vec{w} = 0$ mean geometrically? Why is $\vec{u}$ and $\vec{v}$ being almost perpendicular (about 94°) consistent with a dot product close to 0?

**Your answer:**

*Write your answer here.*

## Part 2: Matrix Multiplication with Loops

Each entry of $C = AB$ is a dot product: $C_{ij} = \sum_k A_{ik} B_{kj}$.

In [ ]:
def my_matmul(A, B):
    # TODO:
    # 1. raise ValueError if the inner dimensions do not match
    # 2. create a zero result of shape (rows of A, columns of B)
    # 3. fill each entry with the dot product of row i of A and column j of B, using loops
    return None

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert np.allclose(my_matmul([[1, 2], [3, 4]], [[5, 6], [7, 8]]), [[19, 22], [43, 50]])   # lesson example
for shape_a, shape_b in [((3, 4), (4, 2)), ((1, 5), (5, 1)), ((5, 1), (1, 5))]:
    A, B = rng.normal(size=shape_a), rng.normal(size=shape_b)
    assert np.allclose(my_matmul(A, B), A @ B)
try:
    my_matmul(np.ones((2, 3)), np.ones((2, 3)))
    raise AssertionError("should have raised ValueError")
except ValueError:
    pass
print("passed")

In [ ]:
# TODO 2.2: time my_matmul vs the @ operator on two random 120 x 120 matrices (use time.perf_counter)

**Question 2.** A (1×5) @ (5×1) and a (5×1) @ (1×5) product use the same numbers. What shapes do they produce, and what are they called?

**Your answer:**

*Write your answer here.*

## Part 3: Matrix Properties and the Inverse

The lesson introduced the inverse briefly. Here you will work with it directly. The **inverse** $A^{-1}$ of a square matrix $A$ satisfies $A A^{-1} = A^{-1} A = I$. It exists only when $A$ is **non-singular**: its determinant is non-zero, or equivalently its columns are **linearly independent** (no column is a combination of the others).

In [ ]:
A = np.array([[2, 1, 0], [1, 3, 1], [0, 1, 4]], dtype=float)
B = np.array([[1, 2, 0], [0, 1, 1], [1, 0, 2]], dtype=float)
I = np.eye(3)

# TODO 3.1: print True/False for each claim
ab_equals_ba = None            # AB == BA ?
transpose_rule = None          # (AB)^T == B^T A^T ?
identity_rule = None           # A I == A ?

# TODO 3.2: compute the inverse of A and check that A @ A_inv is the identity (np.allclose)
A_inv = None

# TODO 3.3: S = [[1, 2], [2, 4]]. Compute its determinant, then try np.linalg.inv(S) inside try/except

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert ab_equals_ba is not None and not ab_equals_ba
assert transpose_rule and identity_rule
assert np.allclose(A @ A_inv, np.eye(3))
print("passed")

### 3.4 Solving a system without the inverse

$$2x + y = 5, \qquad x + 3y = 10 \quad \Longleftrightarrow \quad \begin{bmatrix} 2 & 1 \\ 1 & 3 \end{bmatrix}\begin{bmatrix} x \\ y \end{bmatrix} = \begin{bmatrix} 5 \\ 10 \end{bmatrix}$$

`np.linalg.solve(M, b)` finds the solution directly. It is faster and more accurate than `np.linalg.inv(M) @ b`, which is why numerical code avoids computing inverses.

In [ ]:
# TODO: solve the system with np.linalg.solve and check it by substituting back
xy = None
print(xy)

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert np.allclose(xy, [1, 3])
print("passed")

**Question 3.** Why can't $S$ be inverted? Describe it in terms of its columns.

**Your answer:**

*Write your answer here.*

## Part 4: Cosine Similarity Between Cars

The lesson compared houses using cosine similarity on standardised features. Here you will see what goes wrong without those two steps. A **raw** dot product favours rows with big numbers, not similar rows. **Cosine similarity** divides out the lengths and keeps only direction:

$$\cos(\theta) = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\|\,\|\vec{b}\|}$$

On standardised data it asks: *is this car above or below average in the same features as the target car?*

In [ ]:
features = ["mpg", "cylinders", "displacement", "horsepower", "weight", "acceleration"]
X_raw = cars[features].to_numpy(dtype=float)
target_idx = cars.index[cars["name"] == "ford pinto"][0]
print(f"Target car: row {target_idx}")
cars.loc[target_idx, ["name"] + features]

In [ ]:
def cosine_similarity(a, B):
    # TODO: cosine similarity between vector a and EVERY row of matrix B (return a 1-D array)
    # Use one matrix-vector product and np.linalg.norm(..., axis=1). No loops.
    return None


# TODO 4.2: standardise X_raw column by column (z-scores: subtract the mean, divide by the std)
X_std = None

# TODO 4.3: top 5 most similar cars to the target (excluding itself) by
#   (a) raw dot product on X_raw, and (b) cosine similarity on X_std.
# Show name, mpg, cylinders, weight for each list.

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert np.isclose(cosine_similarity(np.array([1.0, 0]), np.array([[2.0, 0], [0, 3.0]]))[0], 1)
assert np.isclose(cosine_similarity(np.array([1.0, 0]), np.array([[2.0, 0], [0, 3.0]]))[1], 0)
print("passed")

**Question 4.** Compare the two lists. Why does the raw dot product pick those cars?

**Your answer:**

*Write your answer here.*

## Part 5: The Covariance Matrix Is One Matrix Product

If $X_c$ is the data with each column's mean subtracted (centred), then

$$\text{Cov} = \frac{1}{n - 1} X_c^T X_c$$

Entry $(i, j)$ is the dot product of centred column $i$ with centred column $j$, divided by $n-1$: exactly Monday's covariance formula.

In [ ]:
# TODO: build the covariance matrix of X_raw with one matrix product, and compare it with cars[features].cov()
cov_matrix = None

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert np.allclose(cov_matrix, cars[features].cov().to_numpy())
assert np.allclose(cov_matrix, cov_matrix.T)
print("passed: and the matrix is symmetric, as the lesson said")

## Part 6: Normal Equations, Done Properly

Predict `mpg` from six features. Keep the two habits from the lesson:

1. **Split first, then scale.** If you fit `StandardScaler` on *all* the data before splitting, the test set's mean and standard deviation leak into training. Fit the scaler on the training set only, then apply it to both sets.
2. **Solve, don't invert.** Use `np.linalg.solve(XtX, Xty)` instead of `np.linalg.inv(XtX) @ Xty`.

In [ ]:
predictors = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]
X = cars[predictors].to_numpy(dtype=float)
y = cars["mpg"].to_numpy()

# TODO 6.1: train/test split (test_size=0.2, random_state=42), THEN fit a StandardScaler on X_train only
X_train_s, X_test_s, y_train, y_test = None, None, None, None

# TODO 6.2: design matrices with a leading column of ones
def add_intercept(X):
    return None

Xd_train, Xd_test = None, None

# TODO 6.3: weights via np.linalg.solve on the normal equations
w = None

In [ ]:
# TODO 6.4: compare w with (a) np.linalg.lstsq(Xd_train, y_train, rcond=None)[0] and (b) sklearn LinearRegression
# TODO 6.5: test R^2 and RMSE
# TODO 6.6: geometry check. The training residuals should be orthogonal to every column of Xd_train: print Xd_train.T @ residuals

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert np.allclose(w, w_lstsq) and np.allclose(w, w_sklearn)
assert np.allclose(Xd_train.T @ (y_train - Xd_train @ w), 0, atol=1e-8)
print("passed")

**Question 6.** Which standardised coefficients are largest in magnitude? `horsepower` and `displacement` get tiny weights (displacement's is even positive), although both are strongly negatively correlated with mpg. How can that be?

**Your answer:**

*Write your answer here.*

## Part 7: When $X^TX$ Breaks: Collinearity

The lesson mentions that $X^TX$ can be singular "with collinear features". Watch it happen: add `weight` again, converted to kilograms. It is the same information, so the new column is an exact multiple of an existing one.

In [ ]:
# TODO 7.1: add a weight_kg column (weight * 0.4536) to the training features, standardise (fit on train), add the intercept
Xd_bad = None

# TODO 7.2: print the number of columns, np.linalg.matrix_rank(Xd_bad) and np.linalg.cond(Xd_bad.T @ Xd_bad)

# TODO 7.3: try np.linalg.solve on the normal equations (inside try/except), then use np.linalg.lstsq instead.
# What weights does lstsq give the two weight columns, and what are its predictions compared with the good model?

**Question 7.** Explain what you saw. Why can't the data decide how to split the weight effect between the two columns?

**Your answer:**

*Write your answer here.*

## Part 8: Wrap-Up

In 3 to 4 sentences, explain to a classmate why "multiply the data matrix by a weight vector" is the same thing as "compute a dot product for every row", and where $X^T$ shows up when fitting the model.

**Your answer:**

*Write your answer here.*

> Submit your completed notebook to the Kanban board under your name by end of day.